In [ ]:
"""
Merges all JSON files in ../data/100_contract_dataset/keyphrases/
into a single ../data/100_contract_dataset/extracted_keyphrases.json

Supports two formats:
  - Each file is a JSON list  → final output is one flat merged list
  - Each file is a JSON dict  → final output is one merged dict (later files
                                win on duplicate keys, with a warning)
"""

import json
import sys
from pathlib import Path

INPUT_DIR   = Path("../data/100_contract_dataset/keyphrases")
OUTPUT_FILE = Path("../data/100_contract_dataset/extracted_keyphrases.json")


def merge_json_files(input_dir: Path, output_file: Path) -> None:
    files = sorted(input_dir.glob("*.json"))
    if not files:
        print(f"[ERROR] No JSON files found in {input_dir}", file=sys.stderr)
        sys.exit(1)

    print(f"Found {len(files)} file(s) in {input_dir}")

    merged_list: list = []
    merged_dict: dict = {}
    mode: str | None = None   # "list" or "dict"

    for path in files:
        with open(path, encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError as e:
                print(f"[WARNING] Skipping {path.name}: JSON parse error — {e}", file=sys.stderr)
                continue

        file_mode = "list" if isinstance(data, list) else "dict"

        if mode is None:
            mode = file_mode
        elif mode != file_mode:
            print(
                f"[ERROR] Mixed formats: previous files were '{mode}' "
                f"but {path.name} is '{file_mode}'.",
                file=sys.stderr,
            )
            sys.exit(1)

        if mode == "list":
            merged_list.extend(data)
            print(f"  {path.name}: {len(data)} item(s)  (running total: {len(merged_list)})")
        else:
            overlap = set(data.keys()) & set(merged_dict.keys())
            if overlap:
                print(f"  [WARNING] {path.name}: {len(overlap)} duplicate key(s) — will overwrite: {list(overlap)[:5]}")
            merged_dict.update(data)
            print(f"  {path.name}: {len(data)} key(s)  (running total: {len(merged_dict)})")

    result = merged_list if mode == "list" else merged_dict

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    total = len(result)
    print(f"\nDone. Merged {total} {'items' if mode == 'list' else 'keys'} → {output_file}")


if __name__ == "__main__":
    merge_json_files(INPUT_DIR, OUTPUT_FILE)